In [2]:
import tiktoken
import numpy as np
# from senten

In [3]:
def softmax(x, axis= 0):
    # subtract max for numerical stability

    # across rows
    x_shifted = x - np.max(x, axis = axis, keepdims=True)
    
    exp_x = np.exp(x_shifted)
    
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

In [4]:
# axis= 0 # column
# axis= 1 # row

In [5]:
def self_attention(query , key, value):
    # q, k, v are np arrays and are of same order
    # writing codea asuming they are horizontal matrices
    dk = key.shape[1]
    a = (query @ key.T) / np.sqrt(dk) # dot product but for a number of words will be orchestrsted by cross product

    a = softmax(a , axis = 1)
    
    b = a @ value
    
    return b
    

In [6]:
class MultiHeadAttention:

    def __init__(self, embed_dim, num_heads):
        self.embed_dim = embed_dim 
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.WQ = [np.random.randn(embed_dim, self.head_dim) for _ in range(num_heads)]
        self.WK = [np.random.randn(embed_dim, self.head_dim) for _ in range(num_heads)]
        self.WV = [np.random.randn(embed_dim, self.head_dim) for _ in range(num_heads)]

        self.WO = np.random.randn(embed_dim, embed_dim)

    def self_attention(self,query , key, value):
        a = (query @ key.T) / np.sqrt(self.head_dim)

        a = softmax(a , axis = 1)
        
        b = a @ value
        
        return b
    
    def forward(self, x):
        heads =  []
        for i in range(self.num_heads):

            q = x @ self.WQ[i]
            k = x @ self.WK[i]
            v = x @ self.WV[i]
            head = self.self_attention(q, k, v)
            heads.append(head)

        context_awared = np.concatenate(heads, axis = 1)

        output = context_awared @ self.WO # without this step all the heads will be separate. But this combines the information of all the heads
        return output

positional encoder

In [133]:
import math
math.ceil(5/2)

3

In [145]:
# based on formula

def positional_encoder_f(pos, embedding_dimension):

    if (embedding_dimension % 2 != 0 ):
        raise ValueError("Embedding dimension must be even.")
    encoded_pos = []    
    for i in range (int((embedding_dimension / 2))) :
        a = np.sin(pos/ np.power(10000, ((2 * i) / embedding_dimension)))
        b = np.cos(pos/ np.power(10000, ((2 * i) / embedding_dimension)))
        encoded_pos += [a,b]

    return encoded_pos

In [146]:
def get_positional_encoding_for_fixed_number_of_words(number_of_words, embedding_dimension):
    encoded_positions = []
    for i in range (number_of_words):
        encoded_positions += [positional_encoder_f(i,embedding_dimension)]
        # print(encoded_positions, "\n")

    # print("\n\n", encoded_positions)
    return np.vstack([encoded_positions])
    
    

layer normalization


In [147]:
class LayerNorm:
    def __init__ (self, embed_dim):

        self.gamma = np.ones(embed_dim) # initializing to 1 because initially no scaling 
        self.beta = np.zeros(embed_dim) # initializing to 0 because initially no shifting

    def forward (self, x):
        m = np.mean(x, axis = 1, keepdims=True)
        s = np.std(x, axis = 1, keepdims=True)

        normalized = (x - m ) / (s + 1e-5) # incase preventing divison by zero

        return self.gamma * normalized + self.beta

feed forward network


In [148]:
import torch
from torch import nn
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [149]:
class FeedForward(nn.Module):
    def __init__(self ,embedding_dimension, hidden_dimension):
        super(FeedForward, self).__init__()
        self.linear1 = nn.Linear(embedding_dimension, hidden_dimension)
        self.linear2 = nn.Linear(hidden_dimension, embedding_dimension)

    def forward(self, x):
        x = self.linear1(x)
        x = torch.relu(x)
        x = self.linear2(x)
        return x

In [193]:
np.array(torch.tensor(np.ndarray((2,3))))

C:\Users\maury\AppData\Local\Temp\ipykernel_22532\2683828090.py:1: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  np.array(torch.tensor(np.ndarray((2,3))))


array([[0., 1., 0.],
       [1., 0., 1.]])

In [230]:
class EncoderBlock:

    def __init__(self, embed_dim, num_heads, d_ff):
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        self.attention = MultiHeadAttention(self.embed_dim, self.num_heads)

        self.norm = LayerNorm(embed_dim)
        self.norm2 = LayerNorm(embed_dim)
        self.ffn = FeedForward(embed_dim, d_ff )

    

    def forward(self, x):

        # x is with positional encoding information

        attention_output = self.attention.forward(x)

        x = x + attention_output

        x = self.norm.forward(x) # learnable beta and gamma

        x_tensor = torch.tensor(x, dtype=torch.float32)

        ff_o = self.ffn.forward(x_tensor)

        x = x + (ff_o).detach().numpy()

        x = self.norm2.forward(x) # learnable beta gamma 2

        return x


In [231]:
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4") # it uses BPE
# print(enc.n_vocab)
tokens = (enc.encode("Hello Mangesh here i am  <|endoftext|>", allowed_special={"<|endoftext|>"}))


In [232]:
vocab_size = enc.n_vocab # it has inbuilt special token <|endoftext|>
embed_dim = 512
embedding_matrix = np.random.randn(vocab_size, embed_dim)

def get_embedidngs_with_pe(tokens):
    embeddings = []
    token_count = len(tokens)
    for t in tokens:
        embeddings += [embedding_matrix[t]]
    embeddings = np.concatenate([embeddings], axis = 0)
    pe = get_positional_encoding_for_fixed_number_of_words(token_count, embed_dim)

    return embeddings + pe

In [233]:
embeddings_pe = get_embedidngs_with_pe(tokens)

In [236]:
embeddings_pe.shape

(8, 512)

In [239]:
encoder = EncoderBlock(512, 8,512)

In [240]:
encoder_output = encoder.forward(embeddings_pe)

In [241]:
encoder_output.shape

(8, 512)

### Decoder Block

In [80]:
def masked_multi_head_attention(embedding_of_words):
    r,c = embedding_of_words.shape
    for i in range(r):
        for j in range(i+1,r):
            if (j < c):
                embedding_of_words[i][j] = 0
    
    masked_multi_head_ = multi_head_attnetion(embedding_of_words)

    return masked_multi_head_

In [ ]:
masked_multi_head_attention(tokens)

AttributeError: 'list' object has no attribute 'shape'

In [38]:
def cross_attention(decoder_query_emb, encoder_output):
    heads = []
    for _ in range(8): 
        q, _, _ = generate_q_k_v(decoder_query_emb)
        _, k, v = generate_q_k_v(encoder_output)
        head = self_attention(q, k, v)
        heads.append(head)

    return np.concatenate(heads, axis=1)


In [39]:
def decoder_block(decoder_input, encoder_output, ff_model):
    masked_attn = masked_multi_head_attention(decoder_input)  
    z1 = decoder_input + masked_attn
    z1_norm = normalize(z1)

    cross_attn = cross_attention(z1_norm, encoder_output)
    z2 = z1_norm + cross_attn
    z2_norm = normalize(z2)

    z2_tensor = torch.tensor(z2_norm, dtype=torch.float32).to(device)
    ffr = ff_model(z2_tensor).detach().cpu().numpy()
    z3 = z2_norm + ffr
    out_norm = normalize(z3)

    return out_norm


In [40]:
def decoder_arch(decoder_inp, encoder_output, ff_model, count):
    x = decoder_inp
    for i in range(count):
        x = decoder_block(decoder_inp, encoder_output, ff_model)
        print(f"Decoder block creaated : {i+1}/{count}")
    return x

In [41]:
class OutputHead(nn.Module):
    def __init__(self, *args, **kwargs):
        super(OutputHead,self).__init__()
        self.projection = nn.Linear(embedding_dim, vocab_size)

    def forward(self,x):
        logit = self.projection(x)
        return torch.log_softmax(logit, dim=-1)